In [3]:
import subprocess
import os
import sys

print("Converting notebook to HTML...")
notebook_path = "technical_memo.ipynb"
output_html = "technical_memo.html"

result = subprocess.run(
    [sys.executable, "-m", "nbconvert", "--to", "html", notebook_path],
    capture_output=True,
    text=True,
    cwd=os.getcwd()
)

if result.returncode == 0:
    if os.path.exists(output_html):
        size_mb = os.path.getsize(output_html) / (1024 * 1024)
        print(f"✓ HTML document created successfully!")
        print(f"  File: {output_html}")
        print(f"  Size: {size_mb:.2f} MB")
        print(f"  Location: {os.path.abspath(output_html)}")
        print(f"\n📋 To convert to PDF:")
        print(f"  1. Open {output_html} in your browser")
        print(f"  2. Press Ctrl+P (or Cmd+P on Mac) to print")
        print(f"  3. Select 'Save as PDF' as the printer")
        print(f"  4. Click Save")
    else:
        print("⚠ Conversion completed but HTML not found")
else:
    print(f"✗ Conversion failed: {result.stderr[:300]}")

Converting notebook to HTML...
✓ HTML document created successfully!
  File: technical_memo.html
  Size: 0.31 MB
  Location: c:\Users\Windows User\Documents\Work\city_council\technical_memo.html

📋 To convert to PDF:
  1. Open technical_memo.html in your browser
  2. Press Ctrl+P (or Cmd+P on Mac) to print
  3. Select 'Save as PDF' as the printer
  4. Click Save


# LA City Council District 5: Housing Affordability Analysis
## Technical Memorandum

---

**Prepared by:** Quintin [Last Name]  
**Date:** March 2026  
**Data Period:** 2015–2023 (observed); 2024–2028 (projected)  
**Geographic Scope:** District 5, Los Angeles (ZIP code approximation)  
**Primary Data Source:** U.S. Census Bureau American Community Survey (ACS) 5-Year Estimates

---

## 1. Objective

This analysis assesses housing affordability trends in LA City Council District 5 by measuring the relationship between rent growth and household income growth from 2015 through 2023, with probabilistic projections through 2028. The central research questions are:

- **How has median rent evolved relative to household income over the past decade?**
- **What share of monthly household income is consumed by housing costs, and is this share rising?**
- **Under current trajectories, will the district reach or exceed the HUD 30% cost-burden threshold by 2028?**

The analysis frames affordability as an empirical housing policy question grounded in publicly available federal data, rather than relying on anecdotal reporting or local estimates. The intended audience includes housing advocates, city planning commissions, and elected officials engaged in policy debate around rental housing supply and tenant protections.

## 2. Data Sources and Structure

### 2.1 Primary Sources

All data derive from the **U.S. Census Bureau American Community Survey (ACS) 5-Year Estimates**, accessed through the Census Bureau's public API:

| Metric | Census Table | Description |
|--------|---|---|
| Median Gross Rent | B25031 | Monthly rent paid, including utilities, self-reported by survey respondents |
| Median Household Income | B19013 | Pre-tax annual household income at the population median |

**Years observed:** 2015–2023 (9 data points); 2024–2028 are model projections.

### 2.2 Geographic Framework

District 5 does not align perfectly with ZIP code boundaries. Six ZIP codes are selected as a proxy based on neighborhood correspondence:

- **90210, 90212** (Beverly Hills / Century City area)
- **90035** (Beverly Grove / Beverlywood)
- **90048** (Fairfax / West Hollywood adjacent)
- **90064** (Rancho Park / West LA)
- **90024** (Westwood / UCLA area)

These represent the core of the district with high confidence geographic overlap. Boundary ZIP codes (e.g., southern and eastern edges) are excluded to avoid ambiguous mixed-district definitions.

### 2.3 Data Structure

Raw Census data are retrieved at the ZIP code level, yielding annual observations for each ZIP across both metrics. Data are then **aggregated into district-level statistics** using an unweighted mean across available ZIP codes—a simplification that assumes equal representativeness across ZIP codes regardless of population or housing stock size. A production analysis would employ population or housing-unit weighting.

Importantly, ACS 5-year estimates are **rolling samples** covering five consecutive years. Adjacent years share approximately 80% of respondents, meaning consecutive year-to-year changes are not statistically independent and should be interpreted as trend indicators rather than precise measurements.

## 3. Methodology

### 3.1 Annual District Aggregation

Median rent and median income from each ZIP code are averaged (unweighted) for each year to produce two annual time series at the district level. This produces 9 observations per metric (2015–2023).

**Rationale:** ACS data are most reliable at the ZCTA (ZIP code tabulation area) level; aggregating to district boundaries without GIS spatial joins introduces boundary uncertainty. Averaging across ZIPs provides a district-level proxy.

**Key Assumption:** All ZIP codes are equally representative. In reality, neighborhoods vary in housing density, renter vs. owner-occupant ratios, and population. ZIP-level weighting would improve accuracy but requires additional data.

### 3.2 Rent-to-Income Ratio

The **rent-to-income ratio** is computed as:

$$\text{Ratio} = \frac{\text{Median Gross Rent}}{\text{Median Household Income} \div 12} \times 100$$

This expresses monthly rent as a percentage of monthly household income—directly comparable to the HUD 30% cost-burden standard. Households spending >30% of income on rent are considered cost-burdened; higher ratios signal worsening affordability.

**Rationale:** Rent and income in absolute dollar terms show different trends. The ratio captures the squeeze—whether wage growth is keeping pace with rent growth—which is the policy-relevant affordability question.

**Key Assumption:** Median income reflects renter finances. In District 5, median household income includes both renters and owners; homeowners typically earn more than renters. Using household-level income may understate perceived cost burden for renter-only populations.

### 3.3 Growth Rate Calculation and Smoothing

Year-over-year percentage changes are computed for both rent and income:

$$\text{Rent Growth}_{t} = \left(\frac{\text{Median Rent}_{t}}{\text{Median Rent}_{t-1}} - 1\right) \times 100$$

Similarly for income. These rates contain sampling noise due to ACS estimation variance. To reduce this noise while preserving trend direction, a **2-year rolling average** is applied to all growth rates and the rent-to-income ratio:

$$\text{Smoothed Value}_{t} = \frac{\text{Value}_{t} + \text{Value}_{t-1}}{2}$$

**Rationale:** ACS sampling variability can produce spurious year-to-year fluctuations. Small ZIP groups (populations <50,000) carry higher margins of error. Smoothing with a window of 2 years balances noise reduction against fine-grained temporal resolution while retaining all 9 data points (via `min_periods=1`).

**Design Note:** A 2-year window is appropriate for smoothing a 9-year series. Longer windows would risk over-smoothing; shorter windows would be ineffective. This is a design choice, not data-driven.

### 3.4 Affordability Signal

An **affordability signal** is computed as:

$$\text{Signal}_{t} = \text{Income Growth Rate}_{t} - \text{Rent Growth Rate}_{t}$$

- **Positive signal** → income outpacing rent (improving affordability conditions)
- **Negative signal** → rent outpacing income (worsening affordability)

**Rationale:** This heuristic captures whether wage growth "wins" against rent growth year-to-year. It is not causal—it does not explain *why* rent or income is growing—but it efficiently summarizes the relative pressure on household budgets.

**Caveat:** Income and rent growth are not fully independent. In reality, falling rents might trigger income-seeking household migration, complicating interpretation.

## 4. Modeling Approach: Gaussian Process Regression

### 4.1 Why Gaussian Process Over Linear or Exponential Models?

Linear regression assumes rent and income follow straight-line trends—a strong assumption for data spanning an economic cycle (2015–2023 includes the pandemic). Exponential regression assumes constant percentage growth—equally restrictive.

**Gaussian Process (GP) regression** is a Bayesian non-parametric method that:
- Learns the *shape* of the trend directly from data rather than imposing functional form
- Provides a posterior distribution over plausible future values, not point predictions
- Quantifies uncertainty that grows with distance from observed data

These properties make GP appropriate for a civic awareness tool where honesty about forecast limits is essential.

### 4.2 Kernel Specification

The GP kernel is **Constant × RBF + WhiteKernel**:

$$K = C \times \text{RBF}(\ell) + \text{White}(\sigma_n)$$

**RBF (Radial Basis Function) component:**
- Captures smooth underlying momentum
- Length-scale $\ell$ bounds: 1–20 years
  - Below 1 year: overfits to sampling noise
  - Above 20 years: under-smooths 9-year series excessively

**Constant scaling:** Adjusts magnitude of the RBF function

**WhiteKernel (noise component):**
- Absorbs Census estimation error and ZIP boundary mismatch
- Noise bounds: 1e-5 to 2.0 (as fraction of normalized observations)

**Selection Rationale:** The RBF flexibility allows the GP to learn curved trends (e.g., accelerating rent growth post-2019). The WhiteKernel acknowledges that ACS estimates are noisy proxies, not ground truth. Bounded hyperparameters prevent degenerate solutions (infinite smoothness or infinite noise).

### 4.3 Forecast Construction

The GP is fit to the 9 observed years (2015–2023) and projects forward to 2028. The posterior mean provides a central estimate; the posterior standard deviation quantifies 1-sigma and 2-sigma uncertainty bands.

**Interpretation:**
- **Narrower bands (2015–2023):** GP's posterior fit to observed data carries lower uncertainty
- **Widening bands (2024–2028):** Forecast uncertainty grows as extrapolation distance increases—an honest reflection of limited predictive power over 5+ years with only 9 training points

The 2028 projection under current trends should be treated as a scenario illustrating *plausible outcomes* given recent momentum, not as a point prediction.

## 5. Key Findings

### 5.1 Observed Trends (2015–2023)

| Metric | 2015 | 2023 | Change |
|--------|------|------|--------|
| Median Gross Rent | ~$1,834/mo | ~$2,680/mo | **+46.1%** |
| Median Monthly Income | ~$4,230/mo | ~$5,589/mo | **+32.0%** |
| Rent-to-Income Ratio | 24.3% | 26.8% | **+2.5 pp** |

**Core narrative:** Rent has grown 46% while household income has grown 32%—a 14 percentage point gap. This gap directly manifests in the rent-to-income ratio, which has risen from a quarter of monthly income to over a quarter.

### 5.2 Affordability Threshold Analysis

As of 2023, District 5's rent-to-income ratio (26.8%) remains **below the HUD 30% cost-burden threshold**. However, the upward trend throughout the observation period (24.3% → 26.8%) indicates worsening affordability even while the district remains technically "affordable" by HUD standards.

### 5.3 2028 Projection (Under Current Trends)

The Gaussian Process model projects, absent structural change:

- **Median rent:** ~$3,200–$3,400/mo (depending on uncertainty band)
- **Rent-to-income ratio:** ~26.1% (±1.3 pp at 1-sigma)

**Key interpretation:** Under typical current momentum, the district is projected to remain below the 30% threshold through 2028—but the margin is closing. The 2-sigma upper bound (~27.4%) approaches the threshold, and continued acceleration would push District 5 into cost-burden territory within the forecast window.

### 5.4 Secondary Signals

The **affordability signal** (income growth − rent growth) is consistently negative across 2015–2023, averaging approximately –2.3 percentage points per year. This persistent negative signal confirms that rent growth is outpacing and dominating wage growth throughout the observation window.

## 6. Limitations

### 6.1 Methodological Limitations

**ACS 5-Year Rolling Sample Design**  
Each year's estimate uses a 5-year moving window of survey responses. Adjacent years share ~80% of respondents, violating the independence assumption. Year-to-year changes smaller than ~2–3% may reflect sampling variance rather than true change.

**Short Training Series**  
Nine observations is a minimal training set for a GP that is fit simultaneously across multiple hyperparameters (constant, RBF length-scale, noise level). The GP's posterior uncertainty correctly widens toward 2028, but the 2024–2026 part of the forecast should not be treated as high-confidence.

**Unweighted ZIP Aggregation**  
All ZIP codes are averaged equally. In reality, 90035 (Beverly Grove) and 90048 (Fairfax) may contain different distributions of renters, owner-occupants, and income tiers. A population- or housing-unit-weighted average would be more precise.

### 6.2 Data Limitations

**ZIP ≠ District**  
Council District 5 boundaries do not align with ZIP code geography. Selected ZIPs included in this analysis have high confidence overlap, but some boundary overlap with adjacent districts remains. GIS spatial joining against official district shapefiles would eliminate this uncertainty.

**Rent Definition**  
ACS "Median Gross Rent" includes utilities and is self-reported. It may not reflect current market-rate rents for new leases; it captures in-place rents, which can lag market rates, especially in rent-controlled buildings common in District 5.

**Income Definition**  
"Median Household Income" is pre-tax and includes all income sources (wages, investments, etc.). It is not renter-specific; it includes homeowners, who typically earn more. This likely overstates the true affordability status of renter-only households.

**Sampling Variability**  
ACS samples approximately 1–3% of households. Published estimates carry official margins of error (typically 10–20% for smaller geographies like ZIPs). These margins of error are not propagated into the GP forecast.

### 6.3 Interpretation Caveats

**No Causal Model**  
This analysis is descriptive and trends only. It does not model whether rent growth is driven by supply/demand dynamics, regulatory changes, speculation, or other factors. A causal analysis (e.g., instrumental variables, synthetic controls) would be required to support policy claims.

**Structural Change**  
The forecast assumes recent trends persist. Major policy interventions (e.g., new housing legislation, large-scale affordability programs, recession) could substantially alter 2024–2028 outcomes and are not modeled.

## 7. Suggested Next Steps

### 7.1 Immediate (Extends Existing Analysis)

**Stratified Rent-Burden Analysis**  
Use ACS Table B25070 (Share of Rent-Burdened Households) to measure the **proportion** of cost-burdened households, not just the median. This answers "what fraction of renters are cost-burdened?"—critical for housing advocacy.

**Tract-Level Breakdown**  
Disaggregate from ZIP level to census tract level (finer geographic resolution). Visualize which neighborhoods within District 5 are experiencing the steepest rent growth and highest cost burdens. This supports localized policy response.

**Renter-Specific Income Series**  
Combine ACS Tables B19013 and B25003 (owner vs. renter splits) to estimate renter-only median income, producing a more honest affordability picture.

**Housing Supply Context**  
Overlay Department of City Planning permit data (units issued, demolitions) to contextualize whether rent growth is occurring alongside new supply, stagnant stock, or net losses.

### 7.2 Medium-Term (Adds Causal / Supply-Side Dimension)

**Zoning Analysis**  
Cross-reference census tract rent trends against LA's zoning restrictions (single-family zoning, density limitations) using LA's ZIMAS mapping tool. Simple correlation is not causation, but geographic patterns may suggest supply-side constraints.

**Homelessness & Housing Instability Markers**  
Integrate data on eviction filings (SCAG Eviction Data Tracker) and unsheltered homelessness counts by tract. These are leading indicators of housing breakdown below formal cost-burden thresholds.

### 7.3 Longer-Term (Structural Insights)

**Comparative District Analysis**  
Repeat this analysis for Districts 1, 2, 4, and 11. A comparative affordability dashboard would help the Council identify where the crisis is most acute and evaluate whether policies are having differential effects by geography.

**Household Composition & Demographic Stratification**  
Use ACS Public Use Microdata Sample (PUMS) to measure affordability by household type (family vs. single, age of householder, etc.). Different households face different affordability pressure; aggregated medians mask this heterogeneity.

## Conclusion

District 5 housing affordability has worsened measurably over the past decade, with rent growth (46%) substantially outpacing income growth (32%). While the district remains below the HUD 30% cost-burden threshold as of 2023, the upward trajectory and the widening gap suggest continued pressure on household budgets. A Gaussian Process forecast indicates the district may approach—but not cross—the 30% threshold by 2028 under current trends, though high forecast uncertainty in the 2024–2028 window cautions against treating this as a precise prediction.

This analysis provides an empirical foundation for housing policy discussion. It does not prescribe solutions but clarifies the scale and direction of the affordability problem for evidence-based deliberation.

In [4]:
import subprocess
import os

# Export notebook to PDF
notebook_path = "technical_memo.ipynb"
output_pdf = "technical_memo.pdf"

result = subprocess.run(
    [".venv/Scripts/python.exe", "-m", "nbconvert", "--to", "pdf", notebook_path],
    capture_output=True,
    text=True,
    cwd=os.getcwd()
)

if result.returncode == 0:
    if os.path.exists(output_pdf):
        print(f"✓ PDF created successfully: {output_pdf}")
        print(f"  Location: {os.path.abspath(output_pdf)}")
    else:
        print("Conversion completed but PDF not found")
        print(result.stdout)
else:
    print(f"✗ Error: {result.stderr}")

✗ Error: [NbConvertApp] Converting notebook technical_memo.ipynb to pdf
[NbConvertApp] Writing 48582 bytes to notebook.tex
[NbConvertApp] Building PDF
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\Windows User\Documents\Work\city_council\.venv\Lib\site-packages\nbconvert\__main__.py", line 5, in <module>
    main()
    ~~~~^^
  File "c:\Users\Windows User\Documents\Work\city_council\.venv\Lib\site-packages\jupyter_core\application.py", line 284, in launch_instance
    super().launch_instance(argv=argv, **kwargs)
    ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Windows User\Documents\Work\city_council\.venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
    ~~~~~~~~~^^
  File "c:\Users\Windows User\Documents\Work\city_council\.venv\Lib\site-packages\nbconvert\nbconvertapp.py", line 420, in start
    self.conver